In [ ]:
# =============================================================================
# ФИНАЛЬНАЯ ВЕРСИЯ: Загрузка данных TCGA-BRCA (STAR - Counts)
# =============================================================================

import requests
import json
import pandas as pd

# Настройки для вывода DataFrame
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_colwidth', 300)

# --- 1. ФОРМИРОВАНИЕ ЗАПРОСА ---

files_endpt = "https://api.gdc.cancer.gov/files"

# Фильтр, нацеленный на доступные файлы 'STAR - Counts'
filters = {
    "op": "and",
    "content": [
        {"op": "in", "content": {"field": "cases.project.project_id", "value": ["TCGA-BRCA"]}},
        {"op": "in", "content": {"field": "files.data_category", "value": ["Transcriptome Profiling"]}},
        {"op": "in", "content": {"field": "files.data_type", "value": ["Gene Expression Quantification"]}},
        # Вот ключевое изменение: используем STAR - Counts
        {"op": "in", "content": {"field": "analysis.workflow_type", "value": ["STAR - Counts"]}}
    ]
}

# Поля, которые мы хотим получить (включая тип образца для нашей цели)
fields = [
    "file_id",
    "file_name",
    "cases.case_id",
    "cases.samples.sample_type"
]

# Параметры запроса
params = {
    "filters": json.dumps(filters),
    "fields": ",".join(fields),
    "format": "JSON",
    "size": "500"
}

# --- 2. ВЫПОЛНЕНИЕ ЗАПРОСА И СОЗДАНИЕ DATAFRAME ---

print("Отправка запроса к GDC API...")
response = requests.get(files_endpt, params=params)
response_data = response.json()

if response.status_code == 200 and 'data' in response_data and 'hits' in response_data['data'] and len(response_data['data']['hits']) > 0:
    file_hits = response_data['data']['hits']
    print(f"Успешно найдено {len(file_hits)} файлов.")
    
    parsed_data = []
    for file_hit in file_hits:
        case_info = file_hit.get('cases', [{}])[0]
        sample_info = case_info.get('samples', [{}])[0]
        row = {
            'file_id': file_hit.get('file_id'),
            'file_name': file_hit.get('file_name').replace('.gz', ''), # Убираем .gz для удобства
            'case_id': case_info.get('case_id'),
            'sample_type': sample_info.get('sample_type')
        }
        parsed_data.append(row)
        
    df_files = pd.DataFrame(parsed_data)
    
    print("\nТипы образцов в найденных файлах:")
    print(df_files['sample_type'].value_counts())
    
    # --- 3. ФИЛЬТРАЦИЯ ДАННЫХ И СОЗДАНИЕ МАНИФЕСТА ---
    
    target_samples = ["Primary Tumor", "Solid Tissue Normal"]
    df_filtered = df_files[df_files['sample_type'].isin(target_samples)].copy()
    
    print(f"\nОставляем {len(df_filtered)} файлов для задачи (Опухоль vs. Норма).")
    
    # Берем сбалансированную выборку для загрузки
    df_tumor = df_filtered[df_filtered['sample_type'] == "Primary Tumor"].head(10)
    df_normal = df_filtered[df_filtered['sample_type'] == "Solid Tissue Normal"].head(10)
    
    # Проверяем, есть ли у нас достаточно образцов "Норма"
    if len(df_normal) < 10:
        print(f"Внимание: Найдено только {len(df_normal)} образцов 'Solid Tissue Normal'.")

    final_manifest_df = pd.concat([df_tumor, df_normal])

    # Сохраняем метаданные о файлах, которые будем скачивать. Это пригодится позже!
    metadata_path = "../data/metadata.csv"
    final_manifest_df.to_csv(metadata_path, index=False)
    print(f"\nМетаданные для {len(final_manifest_df)} файлов сохранены в: {metadata_path}")
    
    # Сохраняем манифест для GDC-клиента
    manifest_path = "../data/manifest.txt"
    final_manifest_df[['file_id']].to_csv(manifest_path, sep='\t', index=False, header=False)
    print(f"Манифест для загрузки сохранен в: {manifest_path}")

    # --- 4. ИНСТРУКЦИЯ ПО ЗАГРУЗКЕ ---
    print("\n" + "="*50)
    print("!!! ИНСТРУКЦИЯ ПО ЗАГРУЗКЕ !!!")
    download_dir = "../data/raw"
    print(f"1. Убедитесь, что gdc-client доступен и папка {download_dir} существует (mkdir -p {download_dir})")
    print(f"2. Перейдите в терминале в папку: cd {download_dir}")
    print("3. Выполните команду для начала загрузки:")
    # Путь к манифесту из папки raw: ../manifest.txt
    print(f"\n   /path/to/your/gdc-client download -m ../manifest.txt\n")
    print("="*50)

else:
    print("\n!!! ОШИБКА !!!")
    print(f"Запрос не вернул файлы. Статус-код: {response.status_code}")
    print("Ответ сервера:")
    print(response.text)

Отправка запроса к GDC API...
Успешно найдено 500 файлов.

Типы образцов в найденных файлах:
sample_type
Primary Tumor          461
Solid Tissue Normal     38
Metastatic               1
Name: count, dtype: int64

Оставляем 499 файлов для задачи (Опухоль vs. Норма).

Метаданные для 20 файлов сохранены в: ../data/metadata.csv
Манифест для загрузки сохранен в: ../data/manifest.txt

!!! ИНСТРУКЦИЯ ПО ЗАГРУЗКЕ !!!
1. Убедитесь, что gdc-client доступен и папка ../data/raw существует (mkdir -p ../data/raw)
2. Перейдите в терминале в папку: cd ../data/raw
3. Выполните команду для начала загрузки:

   /path/to/your/gdc-client download -m ../manifest.txt



In [9]:
# --- Получаем точный размер файлов перед загрузкой ---

# Читаем метаданные, которые мы только что сохранили
metadata_df = pd.read_csv("../data/metadata.csv")
file_ids_to_download = list(metadata_df['file_id'])

# Формируем фильтр для API, чтобы запросить информацию только об этих файлах
size_filters = {
    "op": "in",
    "content": {
        "field": "files.file_id",
        "value": file_ids_to_download
    }
}

# Запрашиваем только одно поле - размер файла
size_params = {
    "filters": json.dumps(size_filters),
    "fields": "file_size",
    "format": "JSON",
    "size": len(file_ids_to_download)
}

print("Отправка запроса для получения точного размера файлов...")
response = requests.get(files_endpt, params=size_params)

if response.status_code == 200:
    results = response.json()['data']['hits']
    
    # Суммируем размеры всех файлов
    total_size_bytes = sum(f['file_size'] for f in results)
    total_size_mb = total_size_bytes / (1024**2)
    
    print("\n" + "="*30)
    print(f"ТОЧНЫЙ ОБЩИЙ РАЗМЕР 20 ФАЙЛОВ: {total_size_mb:.2f} МБ")
    print("="*30)
else:
    print(f"Не удалось получить размеры. Ошибка: {response.status_code}")

Отправка запроса для получения точного размера файлов...

ТОЧНЫЙ ОБЩИЙ РАЗМЕР 20 ФАЙЛОВ: 80.94 МБ


Успешно найдено 0 файлов.

Типы образцов в найденных файлах:


KeyError: 'sample_type'